# Phase 2 — Spectral Anatomy and Optimal Compression

Dissection of BERT's internal structure through the singular value spectrum
of its 72 linear layers. The analysis reveals **why** certain components
tolerate compression better than others, and leads to an adaptive compression
strategy that beats uniform compression.

- **Part A** — Spectral analysis (weights only, no inference): fingerprints, heatmaps, architecture diagram.
- **Part B** — Performance analysis (with inference): per-emotion vulnerability, Pareto frontier.
- **Part C** — Export all CSVs to `results/notebook2/` for downstream figure generation.

## 0. Setup

In [ ]:
import sys
import os

# Project root (works both on Colab and locally).
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/transformer-structural-compression-v2'
    IN_COLAB = True
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
    IN_COLAB = False

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    compute_singular_value_energy,
    compute_adaptive_ranks,
    get_compression_ratio,
    get_target_layer_names,
    filter_layer_names,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 11

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
# ---- Experiment configuration -------------------------------------------
MODEL_SUBDIR = 'bert-goemotions-23emo-final'
EXCLUDED_EMOTIONS = ['neutral', 'grief', 'nervousness', 'pride', 'relief']
UNIFORM_RANKS = [512, 384, 256, 128, 64]
ENERGY_THRESHOLDS = [0.80, 0.85, 0.90, 0.95, 0.99]

# Recipes with per-component energy thresholds.
MIXED_RECIPES = {
    'Mixed Conservative': {
        'query': 0.90, 'key': 0.90,
        'value': 0.95, 'attention_output': 0.95,
        'ffn_output': 0.95, 'intermediate': 0.99,
    },
    'Mixed Aggressive': {
        'query': 0.85, 'key': 0.85,
        'value': 0.90, 'attention_output': 0.90,
        'ffn_output': 0.90, 'intermediate': 0.95,
    },
}

# ---- Paths --------------------------------------------------------------
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
MODEL_PATH = os.path.join(RESULTS_DIR, MODEL_SUBDIR)
FIGURES_DIR = RESULTS_DIR                                  # PNGs live next to CSVs
EXPORT_DIR = os.path.join(RESULTS_DIR, 'notebook2')        # All CSVs go here

os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f'Model path:  {MODEL_PATH}')
print(f'Figures dir: {FIGURES_DIR}')
print(f'Export dir:  {EXPORT_DIR}')

In [ ]:
# ---- Load fine-tuned model and compute SVD spectrum of all 72 layers ----
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH, problem_type='multi_label_classification',
)
num_labels = baseline_model.config.num_labels
baseline_model.eval().to(device)

print(f'num_labels = {num_labels}')
print('Computing SVD spectrum of 72 encoder layers...')
energy_info = compute_singular_value_energy(baseline_model)
print(f'Layers analysed: {len(energy_info)}')

In [ ]:
# ---- Organise spectral data by component and layer ----------------------
# Column keys use underscores so they match the figure-generation notebook.
COMP_KEYS = ['Query', 'Key', 'Value', 'Attn_Output', 'FFN_Inter', 'FFN_Output']
COMP_PATTERNS = {
    'Query':       'attention.self.query',
    'Key':         'attention.self.key',
    'Value':       'attention.self.value',
    'Attn_Output': 'attention.output.dense',
    'FFN_Inter':   'intermediate.dense',
    'FFN_Output':  'output.dense',
}
COMP_COLORS = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']


def get_component(name):
    if 'attention.self.query'   in name: return 'Query'
    if 'attention.self.key'     in name: return 'Key'
    if 'attention.self.value'   in name: return 'Value'
    if 'attention.output.dense' in name: return 'Attn_Output'
    if 'intermediate.dense'     in name: return 'FFN_Inter'
    if 'output.dense'           in name and 'attention' not in name: return 'FFN_Output'
    return None


def get_layer_idx(name):
    return int(name.split('.')[3])


# Aggregate spectra per component, and three rank matrices (12 layers x 6 comps).
spectra = {c: [] for c in COMP_KEYS}
rank_matrices = {90: np.zeros((12, 6)), 95: np.zeros((12, 6)), 99: np.zeros((12, 6))}

for name, info in energy_info.items():
    comp = get_component(name)
    layer_idx = get_layer_idx(name)
    comp_idx = COMP_KEYS.index(comp)

    spectra[comp].append(info['singular_values'].numpy())
    rank_matrices[90][layer_idx, comp_idx] = info['rank_for_90']
    rank_matrices[95][layer_idx, comp_idx] = info['rank_for_95']
    rank_matrices[99][layer_idx, comp_idx] = info['rank_for_99']

print('Spectral data organised:')
for c in COMP_KEYS:
    n_layers = len(spectra[c])
    n_svs = len(spectra[c][0])
    print(f'  {c:<12s}: {n_layers} layers, {n_svs} singular values each')

---
## Part A — Spectral analysis (no inference)

### A1. Spectral fingerprint per component

Each component type has a different mathematical structure.  The singular
value decay reveals its **intrinsic compressibility**: fast decay means the
information is concentrated in a few singular vectors (highly compressible);
slow decay means the information is spread out (poorly compressible).

In [ ]:
fig, (ax_main, ax_zoom) = plt.subplots(
    1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [2, 1]},
)

for i, comp in enumerate(COMP_KEYS):
    curves = np.array(spectra[comp])
    curves_norm = curves / curves[:, 0:1]           # Normalise each curve by its largest SV.
    mean = curves_norm.mean(axis=0)
    std = curves_norm.std(axis=0)
    x = np.arange(1, len(mean) + 1)

    ax_main.plot(x, mean, color=COMP_COLORS[i], label=comp.replace('_', ' '), linewidth=2.5)
    ax_main.fill_between(x, mean - std, mean + std, color=COMP_COLORS[i], alpha=0.12)

    ax_zoom.plot(x[:200], mean[:200], color=COMP_COLORS[i], linewidth=2.5)
    ax_zoom.fill_between(
        x[:200], (mean - std)[:200], (mean + std)[:200],
        color=COMP_COLORS[i], alpha=0.12,
    )

for rank_val, ls in [(64, ':'), (128, '--'), (256, '-.')]:
    ax_main.axvline(x=rank_val, color='gray', linestyle=ls, alpha=0.4, linewidth=1)
    ax_main.text(rank_val + 3, 0.92, f'r={rank_val}', fontsize=8, color='gray', alpha=0.7)

ax_main.set_xlabel('Singular value index', fontsize=13)
ax_main.set_ylabel(r'Normalised singular value ($\sigma_i / \sigma_1$)', fontsize=13)
ax_main.set_title(
    'Spectral fingerprint per component' + '\n' + r'(mean $\pm$ 1 std over 12 layers)',
    fontsize=14, fontweight='bold',
)
ax_main.legend(fontsize=11, loc='upper right')
ax_main.set_ylim(-0.02, 1.02)
ax_main.set_xlim(0, 768)

ax_zoom.set_xlabel('Singular value index', fontsize=12)
ax_zoom.set_ylabel('Normalised singular value', fontsize=12)
ax_zoom.set_title('Zoom: first 200', fontsize=12, fontweight='bold')
ax_zoom.set_ylim(-0.02, 1.02)
ax_zoom.set_xlim(0, 200)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'spectral_fingerprint.png'), bbox_inches='tight')
plt.show()
print('Saved: spectral_fingerprint.png')

### A2. Compressibility map — BERT's 72 layers

Each cell shows the **minimum rank required to retain a given fraction of
singular energy**.  Low values (green) indicate highly compressible layers;
high values (red) indicate layers that are sensitive to compression.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8))
display_labels = [c.replace('_', ' ') for c in COMP_KEYS]

for ax, threshold_pct in zip(axes, [90, 95, 99]):
    sns.heatmap(
        rank_matrices[threshold_pct], annot=True, fmt='.0f',
        cmap='RdYlGn_r',
        xticklabels=display_labels,
        yticklabels=[f'Layer {i}' for i in range(12)],
        linewidths=0.8, linecolor='white',
        cbar_kws={'label': 'Required rank', 'shrink': 0.8},
        ax=ax,
    )
    ax.set_title(f'Retained energy: {threshold_pct}%', fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

fig.suptitle(
    "BERT-base compressibility map\n(minimum rank per layer and component)",
    fontsize=16, fontweight='bold', y=1.03,
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'compressibility_map.png'), bbox_inches='tight')
plt.show()
print('Saved: compressibility_map.png')

### A3. Architecture diagram — a radiography of BERT

Visual representation of the full encoder, with each component coloured
according to its compressibility.  Think of it as an **MRI of the model**.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ax.set_aspect('auto')
ax.axis('off')

# Layout parameters.
H, GAP_Y = 0.65, 0.18
W, GAP_X = 2.0, 0.15
BLOCK_GAP = 0.8

cmap = plt.cm.RdYlGn_r
rank95 = rank_matrices[95]
vmin, vmax = rank95.min(), rank95.max()
norm = Normalize(vmin=vmin, vmax=vmax)

for layer_idx in range(12):
    y = layer_idx * (H + GAP_Y)
    ax.text(
        -0.8, y + H / 2, f'Layer {layer_idx}', ha='right', va='center',
        fontsize=10, fontweight='bold', family='monospace',
    )

    # Attention: Q, K, V, Out.
    for j in range(4):
        x = j * (W + GAP_X)
        rank = rank95[layer_idx, j]
        color = cmap(norm(rank))
        rect = FancyBboxPatch(
            (x, y), W, H, boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='white', linewidth=2, zorder=2,
        )
        ax.add_patch(rect)
        comp_short = ['Q', 'K', 'V', 'Out'][j]
        ax.text(
            x + W / 2, y + H / 2, f'{comp_short}\n{int(rank)}',
            ha='center', va='center', fontsize=9, fontweight='bold',
            color='white' if rank > (vmin + vmax) / 2 else 'black', zorder=3,
        )

    # FFN: Inter, Out (wider to reflect larger parameter count).
    ffn_x_start = 4 * (W + GAP_X) + BLOCK_GAP
    ffn_w = 2.5
    for j in range(2):
        x = ffn_x_start + j * (ffn_w + GAP_X)
        comp_idx = j + 4
        rank = rank95[layer_idx, comp_idx]
        color = cmap(norm(rank))
        rect = FancyBboxPatch(
            (x, y), ffn_w, H, boxstyle='round,pad=0.05',
            facecolor=color, edgecolor='white', linewidth=2, zorder=2,
        )
        ax.add_patch(rect)
        comp_short = ['Inter', 'Output'][j]
        ax.text(
            x + ffn_w / 2, y + H / 2, f'{comp_short}\n{int(rank)}',
            ha='center', va='center', fontsize=9, fontweight='bold',
            color='white' if rank > (vmin + vmax) / 2 else 'black', zorder=3,
        )

    # Divider between attention and FFN blocks.
    div_x = 4 * (W + GAP_X) + BLOCK_GAP / 2 - 0.05
    ax.plot(
        [div_x, div_x], [y - 0.02, y + H + 0.02],
        color='gray', linewidth=0.8, linestyle=':', alpha=0.5,
    )

top_y = 12 * (H + GAP_Y) + 0.15
attn_center = (3 * (W + GAP_X) + W) / 2
ffn_center = ffn_x_start + (ffn_w + GAP_X + ffn_w) / 2

ax.text(
    attn_center, top_y + 0.5, 'SELF-ATTENTION', ha='center',
    fontsize=16, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.3),
)
ax.text(
    ffn_center, top_y + 0.5, 'FEED-FORWARD', ha='center',
    fontsize=16, fontweight='bold',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.3),
)
ax.text(
    (attn_center + ffn_center) / 2, top_y + 1.5,
    'BERT-base anatomy: compressibility by layer and component',
    ha='center', fontsize=18, fontweight='bold',
)
ax.text(
    (attn_center + ffn_center) / 2, top_y + 1.0,
    'Minimum rank to retain 95% singular energy (green = highly compressible, red = sensitive)',
    ha='center', fontsize=11, color='gray',
)

ax.set_xlim(-1.5, ffn_x_start + 2 * (ffn_w + GAP_X) + 0.5)
ax.set_ylim(-0.5, top_y + 2.2)

sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=30)
cbar.set_label('Required rank (95% energy)', fontsize=12)

plt.savefig(os.path.join(FIGURES_DIR, 'bert_architecture_anatomy.png'), bbox_inches='tight')
plt.show()
print('Saved: bert_architecture_anatomy.png')

### A4. Cumulative energy curves

For three representative layers (early, middle, late) we show how singular
energy accumulates with each singular value.  Curves that rise quickly mean
the layer is compressible at low rank.

In [ ]:
selected_layers = [0, 5, 11]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, layer_idx in zip(axes, selected_layers):
    for i, comp in enumerate(COMP_KEYS):
        for name, info in energy_info.items():
            if get_layer_idx(name) == layer_idx and get_component(name) == comp:
                cum_energy = info['cumulative_energy'].numpy()
                x = np.arange(1, len(cum_energy) + 1)
                ax.plot(
                    x, cum_energy, color=COMP_COLORS[i],
                    label=comp.replace('_', ' '), linewidth=2, alpha=0.9,
                )
                break

    for threshold, ls, alpha in [(0.90, ':', 0.4), (0.95, '--', 0.5), (0.99, '-', 0.3)]:
        ax.axhline(y=threshold, color='gray', linestyle=ls, alpha=alpha, linewidth=1)
        ax.text(750, threshold - 0.015, f'{int(threshold*100)}%',
                fontsize=8, color='gray', ha='right')

    ax.set_xlabel('Rank (k)', fontsize=12)
    ax.set_ylabel('Cumulative energy', fontsize=12)
    ax.set_title(f'Layer {layer_idx}', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 400)
    ax.set_ylim(0.7, 1.005)
    if layer_idx == 0:
        ax.legend(fontsize=9, loc='lower right')

fig.suptitle(
    'Cumulative energy curves per component\n(fast rise = high compressibility)',
    fontsize=15, fontweight='bold', y=1.04,
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'cumulative_energy_curves.png'), bbox_inches='tight')
plt.show()
print('Saved: cumulative_energy_curves.png')

### A5. Adaptive compression — rank assignment by energy threshold

Rather than a fixed rank for every layer, we assign each layer the minimum
rank that retains a given fraction of singular energy.  The distribution of
assigned ranks shifts as we tighten the threshold.

In [ ]:
adaptive_configs = {}
for t in ENERGY_THRESHOLDS:
    ranks = compute_adaptive_ranks(energy_info, energy_threshold=t)
    adaptive_configs[t] = ranks
    vals = list(ranks.values())
    print(
        f'Threshold {t:.0%}: mean rank = {np.mean(vals):6.1f}, '
        f'min = {min(vals):3d}, max = {max(vals):3d}'
    )

In [ ]:
# Violin + per-component mean rank.
fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(18, 7), gridspec_kw={'width_ratios': [1.3, 1]},
)

# --- Left panel: violin plot of assigned ranks per threshold -------------
violin_data = [list(adaptive_configs[t].values()) for t in ENERGY_THRESHOLDS]
parts = ax1.violinplot(
    violin_data, positions=range(len(ENERGY_THRESHOLDS)),
    showmeans=True, showmedians=True, showextrema=False,
)
violin_colors = ['#27ae60', '#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(violin_colors[i])
    pc.set_alpha(0.7)
parts['cmeans'].set_color('black')
parts['cmedians'].set_color('navy')

rng = np.random.RandomState(42)
for i, ranks in enumerate(violin_data):
    jitter = rng.normal(0, 0.04, size=len(ranks))
    ax1.scatter(np.full(len(ranks), i) + jitter, ranks,
                c='black', s=8, alpha=0.4, zorder=3)

for r, ls in [(64, ':'), (128, '--'), (256, '-.')]:
    ax1.axhline(y=r, color='gray', linestyle=ls, alpha=0.4)
    ax1.text(len(ENERGY_THRESHOLDS) - 0.6, r + 3, f'r={r}', fontsize=8, color='gray')

ax1.set_xticks(range(len(ENERGY_THRESHOLDS)))
ax1.set_xticklabels([f'{t:.0%}' for t in ENERGY_THRESHOLDS], fontsize=12)
ax1.set_xlabel('Energy threshold', fontsize=13)
ax1.set_ylabel('Assigned rank', fontsize=13)
ax1.set_title('Adaptive rank distribution', fontsize=14, fontweight='bold')

# --- Right panel: mean rank per component and threshold ------------------
comp_means = np.zeros((len(COMP_KEYS), len(ENERGY_THRESHOLDS)))
for j, t in enumerate(ENERGY_THRESHOLDS):
    for name, rank in adaptive_configs[t].items():
        comp_means[COMP_KEYS.index(get_component(name)), j] += rank
    comp_means[:, j] /= 12

for i, comp in enumerate(COMP_KEYS):
    ax2.plot(
        ENERGY_THRESHOLDS, comp_means[i, :], 'o-',
        color=COMP_COLORS[i], label=comp.replace('_', ' '),
        linewidth=2.5, markersize=8,
    )

ax2.set_xlabel('Energy threshold', fontsize=13)
ax2.set_ylabel('Mean assigned rank', fontsize=13)
ax2.set_title('Mean rank per component', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xticks(ENERGY_THRESHOLDS)
ax2.set_xticklabels([f'{t:.0%}' for t in ENERGY_THRESHOLDS])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'adaptive_rank_distribution.png'), bbox_inches='tight')
plt.show()
print('Saved: adaptive_rank_distribution.png')

### A6. Subspace overlap between Q, K and V

Do the Query, Key and Value matrices of the same layer look along the same
directions of the input space?  We measure the **subspace overlap** of the
dominant right singular subspaces of Q, K and V.

A high overlap would indicate redundancy that could be exploited with joint
compression (shared Q/K/V factorisation).  A low overlap confirms that each
component captures distinct information and justifies compressing them
independently.

In [ ]:
def compute_subspace_overlap(W1, W2, rank):
    '''Normalised overlap of the top-`rank` right singular subspaces.
    Returns a value in [0, 1]: 1 = identical subspaces, 0 = orthogonal.'''
    _, _, Vh1 = torch.linalg.svd(W1.float(), full_matrices=False)
    _, _, Vh2 = torch.linalg.svd(W2.float(), full_matrices=False)
    sub1 = Vh1[:rank, :]
    sub2 = Vh2[:rank, :]
    M = sub1 @ sub2.T
    return (M ** 2).sum().item() / rank


OVERLAP_RANKS = [32, 64, 128]
OVERLAP_PAIRS = [
    ('Q-K', 'attention.self.query', 'attention.self.key'),
    ('Q-V', 'attention.self.query', 'attention.self.value'),
    ('K-V', 'attention.self.key',   'attention.self.value'),
]
overlap_results = {r: np.zeros((12, 3)) for r in OVERLAP_RANKS}
named_params = dict(baseline_model.named_parameters())

for layer_idx in range(12):
    for pair_idx, (_, pat1, pat2) in enumerate(OVERLAP_PAIRS):
        prefix = f'bert.encoder.layer.{layer_idx}.'
        W1 = named_params[prefix + pat1 + '.weight'].data
        W2 = named_params[prefix + pat2 + '.weight'].data
        for rank in OVERLAP_RANKS:
            overlap_results[rank][layer_idx, pair_idx] = compute_subspace_overlap(W1, W2, rank)

# --- Heatmap per subspace rank -------------------------------------------
fig, axes = plt.subplots(1, len(OVERLAP_RANKS), figsize=(6 * len(OVERLAP_RANKS), 8), sharey=True)
pair_labels = [p[0] for p in OVERLAP_PAIRS]

for ax, rank in zip(axes, OVERLAP_RANKS):
    sns.heatmap(
        overlap_results[rank], annot=True, fmt='.2f',
        cmap='YlOrRd', vmin=0, vmax=1,
        xticklabels=pair_labels,
        yticklabels=[f'Layer {i}' for i in range(12)],
        linewidths=0.8, linecolor='white',
        cbar_kws={'label': 'Overlap', 'shrink': 0.8},
        ax=ax,
    )
    ax.set_title(f'Top-{rank} subspace', fontsize=13, fontweight='bold')

fig.suptitle(
    'Subspace overlap between Q, K and V\n'
    '(1.0 = identical subspaces, 0.0 = orthogonal)',
    fontsize=15, fontweight='bold', y=1.04,
)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'qkv_subspace_overlap.png'), bbox_inches='tight')
plt.show()

# --- Summary stats -------------------------------------------------------
print('Mean overlap per pair (top-64 subspace):')
for pair_idx, (pair_name, _, _) in enumerate(OVERLAP_PAIRS):
    values = overlap_results[64][:, pair_idx]
    print(
        f'  {pair_name}: {values.mean():.3f} +/- {values.std():.3f}  '
        f'(min={values.min():.3f}, max={values.max():.3f})'
    )

print('\nMean overlap per layer depth (top-64, average over Q-K/Q-V/K-V):')
for layer_idx in range(12):
    mean_overlap = overlap_results[64][layer_idx, :].mean()
    bar = '#' * int(mean_overlap * 40)
    print(f'  Layer {layer_idx:2d}: {mean_overlap:.3f}  {bar}')

print('\nSaved: qkv_subspace_overlap.png')

---
## Part B — Performance analysis (with inference)

### B1. Dataset and evaluator

In [ ]:
_, _, test_ds, emotion_names, data_collator = load_goemotions(
    exclude_emotions=EXCLUDED_EMOTIONS,
)
print(f'Test set: {len(test_ds)} examples')
print(f'Active emotions ({len(emotion_names)}): {emotion_names}')

eval_args = TrainingArguments(
    output_dir=os.path.join(RESULTS_DIR, 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)


def evaluate_model(model):
    trainer = Trainer(
        model=model, args=eval_args,
        compute_metrics=compute_metrics, data_collator=data_collator,
    )
    return trainer.evaluate(test_ds)


# Shared containers for all strategies evaluated below.
all_f1_macro = {}      # name -> float
all_ratio = {}         # name -> float (compression ratio)
all_per_emotion = {}   # name -> np.ndarray of length num_labels
mixed_recipe_ranks = {}  # recipe_name -> {layer_name: rank}

### B2. Baseline and uniform compression

In [ ]:
print('Evaluating Baseline...')
metrics_bl = evaluate_model(baseline_model)
all_f1_macro['Baseline'] = metrics_bl['eval_f1_macro']
all_ratio['Baseline'] = 1.0
all_per_emotion['Baseline'] = np.array(
    [metrics_bl[f'eval_f1_label_{i}'] for i in range(num_labels)]
)
print(f'  F1 macro: {all_f1_macro["Baseline"]:.4f}')

for rank in UNIFORM_RANKS:
    name = f'Uniform r={rank}'
    print(f'Evaluating {name}...')
    model = apply_svd_compression(baseline_model, rank=rank)
    model.to(device)
    metrics = evaluate_model(model)

    all_f1_macro[name] = metrics['eval_f1_macro']
    all_ratio[name] = get_compression_ratio(baseline_model, model)
    all_per_emotion[name] = np.array(
        [metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
    )
    print(f'  F1 macro: {all_f1_macro[name]:.4f}, ratio: {all_ratio[name]:.2f}x')

    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

baseline_f1_macro = all_f1_macro['Baseline']
print(f'\nTotal evaluated: {len(all_f1_macro)} models')

### B3. Adaptive compression

In [ ]:
for t in ENERGY_THRESHOLDS:
    name = f'Adaptive {t:.0%}'
    print(f'Evaluating {name}...')
    model = apply_svd_compression(baseline_model, rank=adaptive_configs[t])
    model.to(device)
    metrics = evaluate_model(model)

    all_f1_macro[name] = metrics['eval_f1_macro']
    all_ratio[name] = get_compression_ratio(baseline_model, model)
    all_per_emotion[name] = np.array(
        [metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
    )
    print(f'  F1 macro: {all_f1_macro[name]:.4f}, ratio: {all_ratio[name]:.2f}x')

    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

print(f'\nTotal evaluated: {len(all_f1_macro)} models')

### B4. Per-emotion vulnerability

Does compression hit every emotion the same way?  We evaluate F1 per emotion
at every compression level, then sort by the drop induced by the most
aggressive rank.

In [ ]:
baseline_per_emotion = all_per_emotion['Baseline']
compare_keys = ['Baseline'] + [f'Uniform r={r}' for r in UNIFORM_RANKS]
f1_matrix = np.array([all_per_emotion[k] for k in compare_keys]).T

f1_drop = baseline_per_emotion - all_per_emotion[f'Uniform r={UNIFORM_RANKS[-1]}']
sort_idx = np.argsort(-f1_drop)
sorted_emotions = [emotion_names[i] for i in sort_idx]
sorted_matrix = f1_matrix[sort_idx, :]

fig, ax = plt.subplots(figsize=(10, 13))
sns.heatmap(
    sorted_matrix, annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=0, vmax=1,
    xticklabels=['Baseline'] + [f'r={r}' for r in UNIFORM_RANKS],
    yticklabels=sorted_emotions,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'F1 score', 'shrink': 0.6},
    ax=ax,
)

# Highlight the top-5 most vulnerable emotions.
for i in range(5):
    ax.get_yticklabels()[i].set_fontweight('bold')
    ax.get_yticklabels()[i].set_color('#c0392b')

ax.set_title(
    'Per-emotion vulnerability under uniform SVD compression\n'
    '(sorted by largest drop, top 5 in red)',
    fontsize=14, fontweight='bold',
)
ax.set_xlabel('Compression level', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'emotion_vulnerability_heatmap.png'), bbox_inches='tight')
plt.show()
print('Saved: emotion_vulnerability_heatmap.png')

In [ ]:
# Radar chart comparing baseline vs. aggressive uniform compressions.
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection='polar'))

angles = np.linspace(0, 2 * np.pi, num_labels, endpoint=False).tolist()
angles += angles[:1]

radar_configs = [
    ('Baseline',      '#2ecc71', 2.5, '-',  0.12),
    ('Uniform r=256', '#3498db', 2.0, '--', 0.06),
    ('Uniform r=128', '#e67e22', 2.0, '--', 0.06),
    ('Uniform r=64',  '#e74c3c', 2.5, '-',  0.08),
]
for name, color, lw, ls, fill_alpha in radar_configs:
    values = all_per_emotion[name].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', color=color, linewidth=lw, linestyle=ls,
            label=name, markersize=3)
    ax.fill(angles, values, color=color, alpha=fill_alpha)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(emotion_names, fontsize=8)
ax.set_ylim(0, 1)
ax.set_rticks([0.2, 0.4, 0.6, 0.8])
ax.set_rlabel_position(30)
ax.set_title('Per-emotion F1 under SVD compression\n', fontsize=16, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'emotion_radar.png'), bbox_inches='tight')
plt.show()
print('Saved: emotion_radar.png')

### B5. Pareto frontier — Uniform vs. Adaptive

We plot every strategy in a single chart.  The Pareto frontier connects the
non-dominated configurations (best available trade-offs).

In [ ]:
def pareto_front(points):
    '''Return the strictly non-dominated subset of (ratio, retention) points.
    A point is dominated if another has >= ratio AND >= retention with at
    least one strict inequality.'''
    pts = sorted(points, key=lambda p: (-p[0], -p[1]))
    frontier = []
    best_ret = -np.inf
    for ratio, ret in pts:
        if ret > best_ret:
            frontier.append((ratio, ret))
            best_ret = ret
    return sorted(frontier, key=lambda p: p[0])


def _scatter_strategy(ax, ratios, retentions, *, label, color, marker, size, annotate):
    ax.scatter(ratios, retentions, c=color, s=size, zorder=5,
               edgecolors='white', linewidth=2, marker=marker, label=label)
    for txt, x, y in annotate:
        ax.annotate(txt, (x, y), textcoords='offset points',
                    xytext=(10, 8), fontsize=10, fontweight='bold', color=color)


fig, ax = plt.subplots(figsize=(12, 8))

uniform_ratios = [all_ratio[f'Uniform r={r}'] for r in UNIFORM_RANKS]
uniform_retentions = [all_f1_macro[f'Uniform r={r}'] / baseline_f1_macro * 100
                      for r in UNIFORM_RANKS]
_scatter_strategy(
    ax, uniform_ratios, uniform_retentions,
    label='Uniform', color='#3498db', marker='o', size=150,
    annotate=[(f'r={r}', x, y) for r, x, y in zip(UNIFORM_RANKS, uniform_ratios, uniform_retentions)],
)

adaptive_ratios = [all_ratio[f'Adaptive {t:.0%}'] for t in ENERGY_THRESHOLDS]
adaptive_retentions = [all_f1_macro[f'Adaptive {t:.0%}'] / baseline_f1_macro * 100
                       for t in ENERGY_THRESHOLDS]
_scatter_strategy(
    ax, adaptive_ratios, adaptive_retentions,
    label='Adaptive', color='#e74c3c', marker='D', size=150,
    annotate=[(f'{t:.0%}', x, y) for t, x, y in zip(ENERGY_THRESHOLDS, adaptive_ratios, adaptive_retentions)],
)

ax.scatter([1.0], [100.0], c='#2ecc71', s=200, zorder=6,
           edgecolors='white', linewidth=2, marker='*', label='Baseline')
ax.annotate('Baseline', (1.0, 100.0), textcoords='offset points',
            xytext=(12, 5), fontsize=11, fontweight='bold', color='#27ae60')

pareto = pareto_front(
    [(1.0, 100.0)]
    + list(zip(uniform_ratios, uniform_retentions))
    + list(zip(adaptive_ratios, adaptive_retentions))
)
ax.plot([p[0] for p in pareto], [p[1] for p in pareto],
        '--', color='gray', alpha=0.5, linewidth=1.5, zorder=1)

ax.fill_between(
    [0.9, max(uniform_ratios + adaptive_ratios) + 0.1],
    [95, 95], [105, 105], alpha=0.05, color='green',
    label='Ideal zone (>95% retention)',
)

ax.set_xlabel('Compression ratio (x)', fontsize=14)
ax.set_ylabel('F1 macro retention (%)', fontsize=14)
ax.set_title(
    'Pareto frontier: uniform vs. adaptive compression\n'
    '(upper-right is better)',
    fontsize=15, fontweight='bold',
)
ax.legend(fontsize=12, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_ylim(min(uniform_retentions + adaptive_retentions) - 5, 102)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pareto_frontier.png'), bbox_inches='tight')
plt.show()
print('Saved: pareto_frontier.png')

### B6. Data-driven mixed recipes

Earlier sections revealed that each component type has a different
compression sensitivity: Q and K are highly compressible (fast spectral
decay), while FFN Intermediate is fragile (flat spectrum).

Adaptive compression already exploits per-layer differences, but applies the
same energy threshold to every component.  The next step is a **mixed
recipe**: different energy thresholds per component type, compressing the
resilient ones aggressively and protecting the fragile ones.

- **Conservative**: more protection for sensitive components (FFN Inter at 99%).
- **Aggressive**: maximise compression while retaining acceptable quality.

In [ ]:
all_target = get_target_layer_names(baseline_model)

for recipe_name, recipe in MIXED_RECIPES.items():
    print('=' * 55)
    print(f'Recipe: {recipe_name}')
    print('=' * 55)

    ranks_dict = {}
    for comp_name, threshold in recipe.items():
        for name in filter_layer_names(all_target, component=comp_name):
            cum_energy = energy_info[name]['cumulative_energy']
            rank = int((cum_energy >= threshold).nonzero(as_tuple=True)[0][0]) + 1
            ranks_dict[name] = rank

    mixed_recipe_ranks[recipe_name] = ranks_dict      # keep for CSV export

    rank_vals = list(ranks_dict.values())
    print(f'Thresholds: { {c: f"{t:.0%}" for c, t in recipe.items()} }')
    print(
        f'Assigned ranks: min={min(rank_vals)}, max={max(rank_vals)}, '
        f'mean={np.mean(rank_vals):.1f}'
    )

    model = apply_svd_compression(baseline_model, rank=ranks_dict)
    model.to(device)
    metrics = evaluate_model(model)

    all_f1_macro[recipe_name] = metrics['eval_f1_macro']
    all_ratio[recipe_name] = get_compression_ratio(baseline_model, model)
    all_per_emotion[recipe_name] = np.array(
        [metrics[f'eval_f1_label_{i}'] for i in range(num_labels)]
    )

    retention = all_f1_macro[recipe_name] / baseline_f1_macro * 100
    print(f'F1 macro: {all_f1_macro[recipe_name]:.4f}')
    print(f'Compression ratio: {all_ratio[recipe_name]:.2f}x')
    print(f'F1 retention: {retention:.1f}%\n')

    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# Extended Pareto: uniform + adaptive + mixed recipes.
fig, ax = plt.subplots(figsize=(12, 8))

_scatter_strategy(
    ax, uniform_ratios, uniform_retentions,
    label='Uniform', color='#3498db', marker='o', size=150,
    annotate=[(f'r={r}', x, y) for r, x, y in zip(UNIFORM_RANKS, uniform_ratios, uniform_retentions)],
)
_scatter_strategy(
    ax, adaptive_ratios, adaptive_retentions,
    label='Adaptive', color='#e74c3c', marker='D', size=150,
    annotate=[(f'{t:.0%}', x, y) for t, x, y in zip(ENERGY_THRESHOLDS, adaptive_ratios, adaptive_retentions)],
)

mixed_names = list(MIXED_RECIPES.keys())
mixed_ratios = [all_ratio[n] for n in mixed_names]
mixed_retentions = [all_f1_macro[n] / baseline_f1_macro * 100 for n in mixed_names]
_scatter_strategy(
    ax, mixed_ratios, mixed_retentions,
    label='Mixed recipe', color='#8e44ad', marker='*', size=250,
    annotate=[(n.replace('Mixed ', ''), x, y)
              for n, x, y in zip(mixed_names, mixed_ratios, mixed_retentions)],
)

ax.scatter([1.0], [100.0], c='#2ecc71', s=200, zorder=6,
           edgecolors='white', linewidth=2, marker='P', label='Baseline')
ax.annotate('Baseline', (1.0, 100.0), textcoords='offset points',
            xytext=(12, 5), fontsize=11, fontweight='bold', color='#27ae60')

pareto = pareto_front(
    [(1.0, 100.0)]
    + list(zip(uniform_ratios, uniform_retentions))
    + list(zip(adaptive_ratios, adaptive_retentions))
    + list(zip(mixed_ratios, mixed_retentions))
)
ax.plot([p[0] for p in pareto], [p[1] for p in pareto],
        '--', color='gray', alpha=0.5, linewidth=1.5, zorder=1)

all_ratios_flat = uniform_ratios + adaptive_ratios + mixed_ratios
all_retentions_flat = uniform_retentions + adaptive_retentions + mixed_retentions
ax.fill_between(
    [0.9, max(all_ratios_flat) + 0.1],
    [95, 95], [105, 105], alpha=0.05, color='green',
    label='Ideal zone (>95% retention)',
)

ax.set_xlabel('Compression ratio (x)', fontsize=14)
ax.set_ylabel('F1 macro retention (%)', fontsize=14)
ax.set_title(
    'Pareto frontier: uniform vs. adaptive vs. mixed recipe\n'
    '(upper-right is better)',
    fontsize=15, fontweight='bold',
)
ax.legend(fontsize=12, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_ylim(min(all_retentions_flat) - 5, 102)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'mixed_recipe_pareto.png'), bbox_inches='tight')
plt.show()
print('Saved: mixed_recipe_pareto.png')
print(f'\nTotal evaluated: {len(all_f1_macro)} models')

### B7. Lollipop — most vulnerable emotions

Quantifies how much F1 each emotion loses under aggressive uniform
compression.  Rare, semantically subtle emotions are the first to collapse.

In [ ]:
worst_rank = min(UNIFORM_RANKS)
drops = all_per_emotion['Baseline'] - all_per_emotion[f'Uniform r={worst_rank}']
sort_idx = np.argsort(drops)
sorted_drops = drops[sort_idx]
sorted_names = [emotion_names[i] for i in sort_idx]
sorted_baseline = all_per_emotion['Baseline'][sort_idx]

fig, ax = plt.subplots(figsize=(10, 12))
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(sorted_drops)))

for i, (name, drop, bl) in enumerate(zip(sorted_names, sorted_drops, sorted_baseline)):
    ax.plot([bl - drop, bl], [i, i], color=colors[i], linewidth=2.5, zorder=2)
    ax.scatter(bl, i, color='#2ecc71', s=60, zorder=3, edgecolors='white', linewidth=1)
    ax.scatter(bl - drop, i, color=colors[i], s=60, zorder=3, edgecolors='white', linewidth=1)

ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('F1 score', fontsize=13)
ax.set_title(
    f'SVD compression impact per emotion (rank {worst_rank})\n'
    'Green dot = baseline, coloured dot = compressed',
    fontsize=14, fontweight='bold',
)
ax.set_xlim(-0.05, 1.0)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71',
           markersize=10, label='Baseline'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c',
           markersize=10, label=f'Rank {worst_rank}'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'emotion_lollipop.png'), bbox_inches='tight')
plt.show()
print('Saved: emotion_lollipop.png')

---
## Part C — Export results to `results/notebook2/`

Every intermediate data structure is written to CSV.  Consumers include
`plots/generate_cap4_figures_ant.ipynb` and any other downstream notebook.

In [ ]:
# ---- 1. Rank matrices (12 layers x 6 components) per energy threshold ---
for t_pct, matrix in rank_matrices.items():
    df = pd.DataFrame(matrix, columns=COMP_KEYS,
                      index=[f'L{i}' for i in range(12)])
    df.index.name = 'layer'
    df.to_csv(os.path.join(EXPORT_DIR, f'rank_matrix_k{t_pct}.csv'))
    print(f'Saved: rank_matrix_k{t_pct}.csv  {df.shape}')

# ---- 2. Aggregated singular values per component (mean +/- std) ---------
sv_rows = []
for comp in COMP_KEYS:
    curves = np.array(spectra[comp])
    curves_norm = curves / curves[:, 0:1]
    mean = curves_norm.mean(axis=0)
    std = curves_norm.std(axis=0)
    for k in range(len(mean)):
        if k >= len(sv_rows):
            sv_rows.append({'rank_index': k + 1})
        sv_rows[k][f'{comp}_mean'] = mean[k]
        sv_rows[k][f'{comp}_std'] = std[k]
sv_df = pd.DataFrame(sv_rows)
sv_df.to_csv(os.path.join(EXPORT_DIR, 'singular_values_by_component.csv'), index=False)
print(f'Saved: singular_values_by_component.csv  {sv_df.shape}')

# ---- 3. Per-layer spectral summary (72 rows) ----------------------------
summary_rows = []
for name, info in energy_info.items():
    sv = info['singular_values'].numpy()
    cum = info['cumulative_energy'].numpy()
    energy = sv ** 2
    total_energy = float(energy.sum())
    # Spectral entropy (base-2) of the normalised energy distribution.
    p = energy / total_energy
    p_pos = p[p > 0]
    entropy_bits = float(-(p_pos * np.log2(p_pos)).sum())
    summary_rows.append({
        'layer_name': name,
        'component': get_component(name),
        'layer_idx': get_layer_idx(name),
        'n_singular_values': int(len(sv)),
        'max_sv': float(sv[0]),
        'min_sv': float(sv[-1]),
        'sum_sv': float(sv.sum()),
        'total_energy': total_energy,
        'rank_for_90': int(info['rank_for_90']),
        'rank_for_95': int(info['rank_for_95']),
        'rank_for_99': int(info['rank_for_99']),
        'spectral_entropy_bits': entropy_bits,
        'effective_rank': float(np.exp(entropy_bits * np.log(2))),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(EXPORT_DIR, 'spectral_per_layer.csv'), index=False)
print(f'Saved: spectral_per_layer.csv  {summary_df.shape}')

# ---- 4. Full raw spectrum (72 * 768 = 55,296 rows) ----------------------
raw_rows = []
for name, info in energy_info.items():
    sv = info['singular_values'].numpy()
    cum = info['cumulative_energy'].numpy()
    sv_max = float(sv[0])
    comp = get_component(name)
    layer_idx = get_layer_idx(name)
    for k in range(len(sv)):
        raw_rows.append({
            'layer_name': name,
            'component': comp,
            'layer_idx': layer_idx,
            'rank_index': k + 1,
            'singular_value': float(sv[k]),
            'singular_value_normalised': float(sv[k] / sv_max),
            'cumulative_energy': float(cum[k]),
        })
raw_df = pd.DataFrame(raw_rows)
raw_df.to_csv(os.path.join(EXPORT_DIR, 'singular_values_raw.csv'), index=False)
print(f'Saved: singular_values_raw.csv  {raw_df.shape}')

In [ ]:
# ---- 5. Per-emotion F1 across all strategies (wide format) --------------
emo_rows = []
for emo_idx, emo in enumerate(emotion_names):
    row = {'emotion': emo}
    for strategy_name, f1_array in all_per_emotion.items():
        row[strategy_name] = f1_array[emo_idx]
    emo_rows.append(row)
emo_wide_df = pd.DataFrame(emo_rows)
emo_wide_df.to_csv(os.path.join(EXPORT_DIR, 'uniform_emotion_f1.csv'), index=False)
print(f'Saved: uniform_emotion_f1.csv  {emo_wide_df.shape}')

# ---- 6. Per-emotion F1 (long format) ------------------------------------
long_rows = []
for name in all_per_emotion:
    for i, emo in enumerate(emotion_names):
        long_rows.append({'strategy': name, 'emotion': emo,
                          'f1': float(all_per_emotion[name][i])})
long_df = pd.DataFrame(long_rows)
long_df.to_csv(os.path.join(EXPORT_DIR, 'per_emotion_results.csv'), index=False)
print(f'Saved: per_emotion_results.csv  {long_df.shape}')

# ---- 7. Adaptive rank distribution (aggregate per threshold/component) --
adaptive_rank_rows = []
for t in ENERGY_THRESHOLDS:
    per_comp = {c: [] for c in COMP_KEYS}
    for name, rank in adaptive_configs[t].items():
        per_comp[get_component(name)].append(rank)
    for comp in COMP_KEYS:
        vals = per_comp[comp]
        adaptive_rank_rows.append({
            'threshold': int(t * 100),
            'component': comp,
            'mean_rank': float(np.mean(vals)),
            'min_rank': int(np.min(vals)),
            'max_rank': int(np.max(vals)),
        })
adaptive_rank_df = pd.DataFrame(adaptive_rank_rows)
adaptive_rank_df.to_csv(
    os.path.join(EXPORT_DIR, 'adaptive_rank_distribution.csv'), index=False,
)
print(f'Saved: adaptive_rank_distribution.csv  {adaptive_rank_df.shape}')

# ---- 8. Adaptive ranks per layer (wide: threshold columns) --------------
adaptive_layer_rows = []
sample_layer_names = list(adaptive_configs[ENERGY_THRESHOLDS[0]].keys())
for name in sample_layer_names:
    row = {
        'layer_name': name,
        'component': get_component(name),
        'layer_idx': get_layer_idx(name),
    }
    for t in ENERGY_THRESHOLDS:
        row[f'rank_t{int(t*100)}'] = int(adaptive_configs[t][name])
    adaptive_layer_rows.append(row)
pd.DataFrame(adaptive_layer_rows).to_csv(
    os.path.join(EXPORT_DIR, 'adaptive_ranks_per_layer.csv'), index=False,
)
print(f'Saved: adaptive_ranks_per_layer.csv  ({len(adaptive_layer_rows)} rows)')

# ---- 9. Mixed recipe ranks per layer ------------------------------------
mixed_layer_rows = []
for name in sample_layer_names:
    row = {
        'layer_name': name,
        'component': get_component(name),
        'layer_idx': get_layer_idx(name),
    }
    for recipe_name, ranks_dict in mixed_recipe_ranks.items():
        col = recipe_name.replace(' ', '_')
        row[f'rank_{col}'] = int(ranks_dict[name])
    mixed_layer_rows.append(row)
pd.DataFrame(mixed_layer_rows).to_csv(
    os.path.join(EXPORT_DIR, 'mixed_recipe_ranks_per_layer.csv'), index=False,
)
print(f'Saved: mixed_recipe_ranks_per_layer.csv  ({len(mixed_layer_rows)} rows)')

In [ ]:
# ---- 10. Uniform compression results ------------------------------------
uni_rows = [{
    'strategy': 'Baseline', 'rank': 768,
    'macro_f1': baseline_f1_macro, 'retention': 1.0, 'compression_ratio': 1.0,
}]
for r in UNIFORM_RANKS:
    name = f'Uniform r={r}'
    uni_rows.append({
        'strategy': name, 'rank': r,
        'macro_f1': all_f1_macro[name],
        'retention': all_f1_macro[name] / baseline_f1_macro,
        'compression_ratio': all_ratio[name],
    })
uni_df = pd.DataFrame(uni_rows).sort_values('rank', ascending=False)
uni_df.to_csv(os.path.join(EXPORT_DIR, 'uniform_compression_results.csv'), index=False)
print(f'Saved: uniform_compression_results.csv  {uni_df.shape}')

# ---- 11. Adaptive compression results -----------------------------------
adaptive_rows = [{
    'strategy': f'Adaptive {t:.0%}',
    'threshold': int(t * 100),
    'macro_f1': all_f1_macro[f'Adaptive {t:.0%}'],
    'retention': all_f1_macro[f'Adaptive {t:.0%}'] / baseline_f1_macro,
    'compression_ratio': all_ratio[f'Adaptive {t:.0%}'],
} for t in ENERGY_THRESHOLDS]
pd.DataFrame(adaptive_rows).to_csv(
    os.path.join(EXPORT_DIR, 'adaptive_compression_results.csv'), index=False,
)
print('Saved: adaptive_compression_results.csv')

# ---- 12. Mixed recipes results ------------------------------------------
mixed_rows = [{
    'strategy': name,
    'macro_f1': all_f1_macro[name],
    'retention': all_f1_macro[name] / baseline_f1_macro,
    'compression_ratio': all_ratio[name],
    'min_rank': int(min(mixed_recipe_ranks[name].values())),
    'max_rank': int(max(mixed_recipe_ranks[name].values())),
    'mean_rank': float(np.mean(list(mixed_recipe_ranks[name].values()))),
} for name in MIXED_RECIPES]
pd.DataFrame(mixed_rows).to_csv(
    os.path.join(EXPORT_DIR, 'mixed_compression_results.csv'), index=False,
)
print('Saved: mixed_compression_results.csv')

# ---- 13. Master table (every strategy in one place) ---------------------
master_rows = []
for name in all_f1_macro:
    if name == 'Baseline':
        stype = 'baseline'
    elif name.startswith('Uniform'):
        stype = 'uniform'
    elif name.startswith('Adaptive'):
        stype = 'adaptive'
    else:
        stype = 'mixed'
    master_rows.append({
        'strategy': name, 'type': stype,
        'macro_f1': all_f1_macro[name],
        'retention': all_f1_macro[name] / baseline_f1_macro,
        'compression_ratio': all_ratio[name],
        'f1_retention_pct': all_f1_macro[name] / baseline_f1_macro * 100,
    })
pd.DataFrame(master_rows).to_csv(
    os.path.join(EXPORT_DIR, 'spectral_analysis_results.csv'), index=False,
)
print(f'Saved: spectral_analysis_results.csv  ({len(master_rows)} rows)')

# ---- 14. QKV subspace overlap -------------------------------------------
overlap_rows = []
for layer_idx in range(12):
    for pair_idx, (pair_name, _, _) in enumerate(OVERLAP_PAIRS):
        row = {'layer': layer_idx, 'pair': pair_name}
        for rank in OVERLAP_RANKS:
            row[f'overlap_top_{rank}'] = overlap_results[rank][layer_idx, pair_idx]
        overlap_rows.append(row)
pd.DataFrame(overlap_rows).to_csv(
    os.path.join(EXPORT_DIR, 'qkv_subspace_overlap.csv'), index=False,
)
print(f'Saved: qkv_subspace_overlap.csv  ({len(overlap_rows)} rows)')

print(f'\nAll CSVs written to {EXPORT_DIR}')

## Summary

In [ ]:
figures = [
    'spectral_fingerprint.png',
    'compressibility_map.png',
    'bert_architecture_anatomy.png',
    'cumulative_energy_curves.png',
    'adaptive_rank_distribution.png',
    'qkv_subspace_overlap.png',
    'emotion_vulnerability_heatmap.png',
    'emotion_radar.png',
    'pareto_frontier.png',
    'mixed_recipe_pareto.png',
    'emotion_lollipop.png',
]
csvs = [
    'rank_matrix_k90.csv',
    'rank_matrix_k95.csv',
    'rank_matrix_k99.csv',
    'singular_values_by_component.csv',
    'spectral_per_layer.csv',
    'singular_values_raw.csv',
    'uniform_emotion_f1.csv',
    'per_emotion_results.csv',
    'adaptive_rank_distribution.csv',
    'adaptive_ranks_per_layer.csv',
    'mixed_recipe_ranks_per_layer.csv',
    'uniform_compression_results.csv',
    'adaptive_compression_results.csv',
    'mixed_compression_results.csv',
    'spectral_analysis_results.csv',
    'qkv_subspace_overlap.csv',
]

print('=' * 65)
print('SUMMARY -- Phase 2: Spectral anatomy and optimal compression')
print('=' * 65)
print(f'Baseline F1 macro: {baseline_f1_macro:.4f}')
print(f'Models evaluated: {len(all_f1_macro)}')
print(f'  - 1 baseline')
print(f'  - {len(UNIFORM_RANKS)} uniform compressions')
print(f'  - {len(ENERGY_THRESHOLDS)} adaptive compressions')
print(f'  - {len(MIXED_RECIPES)} mixed recipes')

best_adaptive = max(
    ((t, all_f1_macro[f'Adaptive {t:.0%}'], all_ratio[f'Adaptive {t:.0%}'])
     for t in ENERGY_THRESHOLDS),
    key=lambda x: x[1],
)
print(
    f'\nBest adaptive: {best_adaptive[0]:.0%} energy -> '
    f'F1={best_adaptive[1]:.4f}, ratio={best_adaptive[2]:.2f}x'
)
print('\nMixed recipes:')
for rname in MIXED_RECIPES:
    ret = all_f1_macro[rname] / baseline_f1_macro * 100
    print(
        f'  {rname:<25s}: F1={all_f1_macro[rname]:.4f}, '
        f'ratio={all_ratio[rname]:.2f}x, retention={ret:.1f}%'
    )

print('\nTop-5 most vulnerable emotions (largest drop at rank 64):')
drops_desc = np.argsort(-drops)
for i in range(5):
    idx = drops_desc[i]
    print(
        f'  {emotion_names[idx]:<15s}: {drops[idx]:+.4f} '
        f'({all_per_emotion["Baseline"][idx]:.3f} -> '
        f'{all_per_emotion[f"Uniform r={worst_rank}"][idx]:.3f})'
    )

print(f'\nFigures ({len(figures)}):')
for fig_name in figures:
    status = 'ok' if os.path.exists(os.path.join(FIGURES_DIR, fig_name)) else 'MISSING'
    print(f'  [{status}] {fig_name}')

print(f'\nCSVs in {EXPORT_DIR} ({len(csvs)}):')
for csv_name in csvs:
    status = 'ok' if os.path.exists(os.path.join(EXPORT_DIR, csv_name)) else 'MISSING'
    print(f'  [{status}] {csv_name}')